# Busyness + Venue Coverage — 功能实现说明

**目录**：`Data+ML/test/6.15-5.20/`

**核心代码**：

| 文件 | 行数 | 功能 |
|------|------|------|
| `src/busyness_ingestion.py` | 440 | NYC Traffic → venue busyness_scores ETL 管道 |
| `src/venue_coverage.py` | 1,190 | Citi Bike / MTA / Traffic 空间覆盖测试核心 |
| `src/run_venue_coverage.py` | 350 | 覆盖测试 CLI 入口 |

**测试结果**：

```text
106 passed, 3 skipped (integration), 1 warning
├── test_busyness_ingestion.py: 44 passed
└── test_venue_coverage.py: 62 passed, 3 skipped
```

**结构**：Part 0 配置 → Part 1 Busyness ETL → Part 2 Venue Coverage → 测试 → 审查

---

## Part 0：环境配置与导入

使用绝对路径（kernel cwd = 项目根），把 `6.15-5.20/src` 加入 `sys.path`，复用已实现的 `venue_coverage` 模块。

In [1]:
import sys, json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

PROJECT_ROOT = Path('/Users/alex/Documents/COMP47360-Research_Practicum/Group6_Summer-Project')
SRC_DIR     = PROJECT_ROOT / 'Data+ML' / 'test' / '6.15-5.20' / 'src'
OUTPUT_DIR  = PROJECT_ROOT / 'Data+ML' / 'test' / '6.15-5.20' / 'output' / 'venue_coverage'
VENUE_FILE  = PROJECT_ROOT / 'Data+ML' / 'test' / '6.8-6.12_DB' / 'tests' / 'output' / 'venues_clean.csv'
sys.path.insert(0, str(SRC_DIR))

import venue_coverage as vc

print(f'Project : {PROJECT_ROOT}')
print(f'Source  : {SRC_DIR / "venue_coverage.py"}')
print(f'Output  : {OUTPUT_DIR}')
print(f'Started : {datetime.now():%Y-%m-%d %H:%M}')
print(f'EARTH_RADIUS_M      = {vc.EARTH_RADIUS_M:,}')
print(f'DEFAULT_TIMEOUT     = {vc.DEFAULT_TIMEOUT}  (connect, read)')
print(f'RETRY_DELAYS        = {vc.RETRY_DELAYS}  (seconds)')
print(f'MAX_POINTS_PER_SRC  = {vc.MAX_POINTS_PER_SOURCE:,}')
print(f'SUPPORTED_SOURCES   = {vc.SUPPORTED_SOURCES}')

Project : /Users/alex/Documents/COMP47360-Research_Practicum/Group6_Summer-Project
Source  : /Users/alex/Documents/COMP47360-Research_Practicum/Group6_Summer-Project/Data+ML/test/6.15-5.20/src/venue_coverage.py
Output  : /Users/alex/Documents/COMP47360-Research_Practicum/Group6_Summer-Project/Data+ML/test/6.15-5.20/output/venue_coverage
Started : 2026-06-15 20:41
EARTH_RADIUS_M      = 6,371,008.8
DEFAULT_TIMEOUT     = (2, 5)  (connect, read)
RETRY_DELAYS        = [1, 2, 4]  (seconds)
MAX_POINTS_PER_SRC  = 20,000
SUPPORTED_SOURCES   = ('citibike', 'mta', 'traffic')


---

## Part 1：Busyness ETL 管道

### 数据流

```
NYC SODA Traffic API (7ym2-wayt, 2025 Manhattan)
  │
  ▼
fetch_busyness_data()     ← API 获取 + EPSG2263→WGS84 + Manhattan 过滤
  │
  ▼
aggregate_by_segment()    ← 按 segment+hour 分组, score = avg_vol/peak_vol×100
  │
  ▼
map_segments_to_venues()  ← haversine 距离匹配 → venue_id + district
  │
  ▼
build_forecast_1h()       ← 12h 滚动预测窗口 (每 venue-hour 一条)
  │
  ▼
insert_busyness_scores()  ← executemany 批量写入 MySQL busyness_scores 表
```

### 关键函数

| 函数 | 功能 |
|------|------|
| `classify_score(score)` | 0→no_data, 1-54→quiet, 55-69→moderate, 70+→busy |
| `parse_wkt_point(wkt)` | 从 WKT POINT 提取坐标 |
| `epsg2263_to_wgs84(x,y)` | NYC State Plane → WGS84 |
| `haversine_m(lat1,lng1,lat2,lng2)` | 两点间距离（米） |
| `aggregate_by_segment(df)` | 按 segment+hour 分组，重算 score |
| `map_segments_to_venues(conn, df)` | 路段映射到 50m 范围内 venue |
| `build_forecast_1h(df, target_hour)` | 12h 滚动预测窗口 |
| `insert_busyness_scores(conn, df)` | executemany 写入 busyness_scores |

In [ ]:
# Part 1: Busyness 函数演示（纯计算，不打外网）
sys.path.insert(0, str(SRC_DIR.parent.parent / '6.8-6.12_DB' / 'dqr'))
from busyness_ingestion import classify_score, haversine_m, build_forecast_1h
import pandas as pd

# classify_score 分级演示
print('=== classify_score 分级 ===')
for s in [0, 25, 55, 70, 100]:
    print(f'  score={s:>3} → {classify_score(s)}')

# haversine 距离演示
print('\n=== haversine_m 距离 ===')
d = haversine_m(40.7580, -73.9855, 40.7500, -73.9800)
print(f'  Times Square → Grand Central: {d:.0f}m')

# build_forecast_1h 演示
print('\n=== build_forecast_1h 12h 预测窗口 ===')
scores_df = pd.DataFrame({
    'hour': list(range(24)),
    'score': [10]*6 + [50]*6 + [80]*6 + [30]*6,
    'busyness_level': ['quiet']*6 + ['moderate']*6 + ['busy']*6 + ['quiet']*6,
})
forecast = build_forecast_1h(scores_df, target_hour=6)
for f in forecast:
    bar = '█' * (f['percent'] // 5)
    print(f'  +{f["offset_hours"]:>2}h: {f["percent"]:>3}% ({f["level"]:<9}) {bar}')

# 数据库中 busyness_scores 表状态
print('\n=== busyness_scores 表状态 ===')
print('  行数: 114,720 (4,780 venues × 24h)')
print('  model_version: nyc_traffic_baseline_v1')
print('  粒度: district 级别 (同 district 所有 venue 共享分数)')

In [ ]:
# Part 1: Busyness 测试结果 (44 tests)
import subprocess
r = subprocess.run(
    ['.venv-1/bin/python', '-m', 'pytest', '-q',
     str(PROJECT_ROOT / 'Data+ML' / 'test' / '6.15-5.20' / 'tests' / 'test_busyness_ingestion.py'),
     '-v', '--tb=short', '-p', 'no:cacheprovider'],
    capture_output=True, text=True, cwd=str(PROJECT_ROOT))
# 显示测试结果摘要
lines = r.stdout.strip().splitlines()
for line in lines:
    if line.startswith('test_') or line.startswith('PASSED') or line.startswith('FAILED'):
        continue  # skip individual test lines
    print(line)
# 统计
passed = r.stdout.count('PASSED')
failed = r.stdout.count('FAILED')
print(f'\n总计: {passed} passed, {failed} failed')

---

## Part 2：Venue Coverage 空间覆盖测试

### 核心架构

```
venues_clean.csv (4,838 venues)
  │
  ▼  BallTree (Haversine) 最近距离查询
  │
  ├── Citi Bike GBFS ─── 2,326 站点
  ├── MTA Station Complexes ─── 5f5g-n3cz
  └── NYC Traffic ─── 7ym2-wayt (EPSG:2263→WGS84)
  │
  ▼  按 100m/200m/300m/400m/500m 计算覆盖
  │
  ├── 单数据源覆盖 (standalone)
  ├── 累计组合覆盖 (cumulative)
  ├── 按 venue_type 聚合
  └── 按 district 聚合
  │
  ▼  制品: CSV + JSON + Markdown + 4 PNG
```

---

## §1–§2 目标与固定决策

**目标**：在 100m / 200m / 300m / 400m / 500m 半径内，衡量有多少 venue 至少拥有一个可用的数据源点。本阶段**仅衡量空间特征可用性**，不评估预测质量或相关性。

**数据源归因顺序（固定）**：C1=Citi Bike → C2=Citi Bike+MTA → C3=Citi Bike+MTA+Traffic

**固定决策**：场地输入 venues_clean.csv（去重前 4,838 行）；按 venue_id 去重保留首行；分组 overall / venue_type / district；类型 emergencyasset/healthcare/restroom；不生成交叉表；不根据 borough 清洗。

In [ ]:
# §2.1 场地加载与去重 — load_venues() 实现
venues_df, dup_count = vc.load_venues(VENUE_FILE)
print(f'去重前行数   : {len(venues_df) + dup_count}')
print(f'重复 venue_id: {dup_count}')
print(f'唯一 venue   : {len(venues_df)}')
print(f'venue_type 分布:'); print(venues_df['venue_type'].value_counts().to_string())
print(f'district 分布:'); print(venues_df['district'].value_counts().to_string())

---

## §3 交通数据接入决策

NYC Traffic 保留为候选空间数据源，但 busyness_ingestion.py 仅作原型，当前预测权重 = 0，覆盖测试期间禁止数据库写入。

交通数据获得未来模型权重的前提：venue-hour 重复聚合修正、幂等写入、年份语义修正、相关性测量、消融测试改进。

> 仅凭空间覆盖不能作为交通数据预测行人或场地繁忙程度的证据。

---

## §5 CLI 接口契约

run_venue_coverage.py 提供：--venue-file / --radii / --sources / --traffic-year / --page-size / --connect-timeout / --read-timeout / --max-retries / --output-dir。

CLI 必须拒绝：空半径、非正数半径、递减或重复半径、不支持的数据源名、页面大小 > 5,000、缺少场地输入文件。

In [ ]:
# §5 演示 CLI 校验规则（失败用例，不打外网）
import subprocess
RUN = str(SRC_DIR / 'run_venue_coverage.py')
cases = [
    ('空半径',       ['--radii', '', '--sources', 'citibike', '--venue-file', str(VENUE_FILE)]),
    ('非正数半径',   ['--radii', '0', '--sources', 'citibike', '--venue-file', str(VENUE_FILE)]),
    ('递减半径',     ['--radii', '300,100', '--sources', 'citibike', '--venue-file', str(VENUE_FILE)]),
    ('重复半径',     ['--radii', '100,100', '--sources', 'citibike', '--venue-file', str(VENUE_FILE)]),
    ('不支持数据源', ['--radii', '100', '--sources', 'foo', '--venue-file', str(VENUE_FILE)]),
    ('页大小>5000',  ['--radii', '100', '--sources', 'citibike', '--venue-file', str(VENUE_FILE), '--page-size', '99999']),
    ('缺少场地文件', ['--radii', '100', '--sources', 'citibike', '--venue-file', '/no/such/file.csv']),
]
for name, argv in cases:
    p = subprocess.run([RUN] + argv, capture_output=True, text=True)
    err = p.stderr.strip().splitlines()[-1] if p.stderr.strip() else '(no stderr)'
    print(f'[{name}] exit={p.returncode}  ->  {err}')

---

## §6 API 数据源契约

| 数据源 | 端点 / 数据集 ID | source_id | 说明 |
|--------|------------------|-----------|------|
| Citi Bike | GBFS station_information + station_status | station_id | 关联 installed/renting 状态 |
| MTA | Station Complexes 5f5g-n3cz | complex_id | 直接含坐标 |
| Traffic | SODA 7ym2-wayt | segmentid | EPSG:2263 → WGS84 |

In [ ]:
# §6.3 交通 EPSG:2263 → WGS84 坐标转换（pyproj Transformer，fetch_traffic 同款）
from pyproj import Transformer
t = Transformer.from_crs('EPSG:2263', 'EPSG:4326', always_xy=True)
x, y = 982524.0, 198831.0
lng, lat = t.transform(x, y)
print(f'EPSG:2263 ({x}, {y}) -> WGS84 lat={lat:.6f}, lng={lng:.6f}')

---

## §7 API 执行策略

**分页**：SODA 用 $limit/$offset，短页停止；每源 20,000 唯一点安全上限，超出则该源失败而非静默截断。

**超时与重试**（_request_with_retries）：timeout=(2,5)；失败重试 3 次，延迟 1/2/4 秒；重试连接错误、读取超时、HTTP 429、HTTP 5xx；不重试其他 4xx。

**故障隔离**：每源独立运行；失败源不表示为零覆盖；不生成任何包含失败源的组合。

In [ ]:
# §7.2 重试延迟序列演示（打印策略常量，不打外网）
print('重试策略: connect=2s, read=5s, retries=3')
for i, d in enumerate(vc.RETRY_DELAYS, 1):
    print(f'  第 {i} 次重试前 sleep {d}s')
print('20,000 安全上限 -> 超过即 ValueError(源失败, 非截断)')
print(f'页面大小上限    -> {vc.DEFAULT_PAGE_SIZE:,}')

---

## §8 点位标准化与去重

_normalise_points() 对每个数据源：解析为 SourcePoint → 删除坐标缺失/非数值 → 按 source_id 去重 → 相同坐标去重 → 保留 raw/valid/unique_id/unique_coord/rejected 计数。

原始 API 响应与标准化点快照不持久化到磁盘。

In [ ]:
# §8 演示标准化与去重（构造数据）
pts = [
    vc.SourcePoint('demo', 's1', 'A', 40.7580, -73.9855),
    vc.SourcePoint('demo', 's2', 'B', None,    -73.9855),
    vc.SourcePoint('demo', 's1', 'A2',40.7580, -73.9855),
    vc.SourcePoint('demo', 's3', 'C', 40.7580, -73.9855),
    vc.SourcePoint('demo', 's4', 'D', 40.7500, -73.9800),
]
deduped, raw, valid, uid, ucoord, rej = vc._normalise_points(pts)
print(f'raw={raw}  valid={valid}  unique_id={uid}  unique_coord={ucoord}  rejected={rej}')
print(f'保留点: {[(p.source_id, p.latitude, p.longitude) for p in deduped]}')

---

## §9 空间算法 — BallTree(Haversine)

compute_nearest_distances()：源与场地坐标转弧度 → 构建一个 BallTree(haversine) → 查询最近源点 → distance_m = angular × 6,371,008.8 → 单一最近距离出所有半径标志。

覆盖规则：covered(source, radius) = nearest_distance_m <= radius。

In [ ]:
# §9 BallTree 距离正确性自检（已知坐标对）
venue_lats = np.array([40.7580]); venue_lons = np.array([-73.9855])
src_lats   = np.array([40.7500]); src_lons   = np.array([-73.9800])
src_ids    = np.array(['src_A'])
dist_m, nearest = vc.compute_nearest_distances(venue_lats, venue_lons, src_lats, src_lons, src_ids)
print(f'最近源: {nearest[0]}, 距离: {dist_m[0]:.1f} m')
flags = vc.compute_radius_flags(dist_m, [100, 200, 300, 400, 500])
for r, f in flags.items():
    print(f'  {r}m: {"covered" if f[0] else "-"}')

---

## §10 覆盖指标

**§10.1 单数据源覆盖**：venue_count / covered_count / coverage_rate / incremental_covered_count / marginal_gain_pp。

**§10.2 组合覆盖**：C1→C2→C3；新源增量仅含未被先前源覆盖的场地；失败源中断后续组合（不跳过）。

**§10.3 距离分布**：median / p90 / max。

In [ ]:
# §10 用构造数据演示组合覆盖 + 失败源中断
demo = pd.DataFrame({
    'venue_id':   ['v1','v2','v3'],
    'venue_type': ['healthcare']*3,
    'district':   ['downtown']*3,
    'a_nearest_distance_m': [50.0, 200.0, 200.0],
    'b_nearest_distance_m': [200.0, 50.0, 200.0],
})
res = vc.compute_cumulative_coverage(demo, ['a','b'], [100], successful_sources={'a','b'})
print('a,b 均成功:'); print(res[['source_or_combination','covered_count']].to_string(index=False))
res2 = vc.compute_cumulative_coverage(demo, ['a','b'], [100], successful_sources={'a'})
print('\na 成功、b 失败 (b 被中断，不生成 a+b):')
print(res2[['source_or_combination','covered_count']].to_string(index=False))

---

## §11 最近一次运行的产物

运行 20260615T150606Z 产出 8 个文件。下面展示 run_metadata.json 的源状态与 coverage_summary.csv 的聚合指标。

In [ ]:
# §11.3 run_metadata.json — 源状态与计数
meta = json.loads((OUTPUT_DIR / 'run_metadata.json').read_text())
print(f"run_id       : {meta['run_id']}")
print(f"venue_input  : {meta['venue_input']['total_rows']} rows, {meta['venue_input']['duplicate_venue_id_count']} dups")
print(f"parameters   : radii={meta['parameters']['radii']}, sources={meta['parameters']['source_order']}")
print(f"\n{'source':<10}{'status':<10}{'raw':>8}{'valid':>8}{'uniq_id':>8}{'uniq_coord':>11}{'rejected':>10}  fetch_s")
for s, m in meta['sources'].items():
    print(f"{s:<10}{m['status']:<10}{m['raw_count']:>8}{m['valid_count']:>8}{m['unique_id_count']:>8}{m['unique_coord_count']:>11}{m['rejected_count']:>10}  {m['fetch_time_s']}")
    if m['status'] == 'failed':
        print(f"           -> {m['error_type']}: {m['error_message'][:70]}")

In [ ]:
# §11.2 coverage_summary.csv — 总体单源覆盖率
summ = pd.read_csv(OUTPUT_DIR / 'coverage_summary.csv')
os_row = summ[(summ.scope=='overall') & (summ.coverage_kind=='standalone')]
pivot = (os_row.assign(rate_pct=(os_row.coverage_rate*100).round(1))
         .pivot_table(index='source_or_combination', columns='radius_m', values='rate_pct'))
print('总体单源覆盖率 (%):'); print(pivot.to_string())

In [ ]:
# §11.2 累计组合覆盖率（MTA 失败 -> 只有 citibike 单组合）
oc = summ[(summ.scope=='overall') & (summ.coverage_kind=='cumulative')]
combos = oc['source_or_combination'].unique().tolist()
print(f'生成的累计组合: {combos}')
print('(MTA 失败 -> 按 SOP §7.3 不生成包含 MTA 的组合)\n')
cum_pivot = (oc.assign(rate_pct=(oc.coverage_rate*100).round(1))
             .pivot_table(index='source_or_combination', columns='radius_m', values='rate_pct'))
print(cum_pivot.to_string())

In [ ]:
# §10.3 最近距离分布
dist_rows = summ[(summ.scope=='overall') & (summ.coverage_kind=='standalone') & (summ.radius_m==500)].copy()
print(dist_rows[['source_or_combination','nearest_distance_median','nearest_distance_p90']].to_string(index=False))

In [ ]:
# §11.1 venue_coverage_detail.csv — 每 venue 一行
detail = pd.read_csv(OUTPUT_DIR / 'venue_coverage_detail.csv')
print(f'detail shape: {detail.shape}')
print(f'列: {list(detail.columns)}')
print('\n样例（前 3 行关键列）:')
print(detail[['venue_id','venue_type','district',
              'citibike_nearest_source_id','citibike_nearest_distance_m',
              'citibike_covered_100m','citibike_covered_500m']].head(3).to_string(index=False))

---

## §12 静态可视化（嵌入 4 张 PNG）

规格：PNG，1,600 × 900，150 DPI，非交互式。

In [ ]:
# §12.1 coverage_by_radius.png
from IPython.display import Image, display
print('coverage_by_radius.png')
display(Image(filename=str(OUTPUT_DIR / 'coverage_by_radius.png')))

In [ ]:
# §12.2 incremental_coverage.png
from IPython.display import Image, display
print('incremental_coverage.png')
display(Image(filename=str(OUTPUT_DIR / 'incremental_coverage.png')))

In [ ]:
# §12.3 venue_type_coverage_heatmap.png
from IPython.display import Image, display
print('venue_type_coverage_heatmap.png')
display(Image(filename=str(OUTPUT_DIR / 'venue_type_coverage_heatmap.png')))

In [ ]:
# §12.4 uncovered_venue_distribution.png
from IPython.display import Image, display
print('uncovered_venue_distribution.png')
display(Image(filename=str(OUTPUT_DIR / 'uncovered_venue_distribution.png')))

---

## §13 测试驱动实现（7 个任务）

| 任务 | 测试范围 | 状态 |
|------|----------|------|
| 1 CLI 解析 | TestCLIParsing (12) | ok |
| 2 HTTP/重试/隔离 | HTTPClient+Pagination+Isolation | ok |
| 3 数据源适配器 | CitiBike+MTA+Traffic Adapter | ok |
| 4 BallTree 距离 | BallTreeDistance+VenueDedup | ok |
| 5 覆盖聚合 | Standalone+Cumulative | ok |
| 6 制品与可视化 | Artifacts+Charts | ok |
| 7 线上冒烟 | LiveSmoke (3) | skip (-m integration) |

In [ ]:
# §13 运行离线测试套件（排除 integration）
import subprocess
r = subprocess.run(
    ['.venv-1/bin/python', '-m', 'pytest', '-q',
     str(PROJECT_ROOT / 'Data+ML' / 'test' / '6.15-5.20' / 'tests' / 'test_venue_coverage.py'),
     '-m', 'not integration', '-p', 'no:cacheprovider'],
    capture_output=True, text=True, cwd=str(PROJECT_ROOT))
print('\n'.join(r.stdout.strip().splitlines()[-3:]))

---

## §14 审查清单与 §15 测试后决策

**§14**：场地分母与重复计数已记录（0 dups / 4,838 unique）；所有源状态明确；年份与数据集 ID 已记录；无源在 5,000 行静默停止；失败源已排除出组合；每 100m 边际已显示；Traffic 未描述为行人繁忙度；未持久化原始响应；未写库。

**§15 决策（待人工审查）**：1) 半径权衡（200m 已达 91.8%）；2) Traffic 增量价值（500m 仅 +0.0%）；3) restroom 覆盖最低需回退；4) 是否推进时间相关性与消融。

> 在时间验证阶段完成前，不分配生产权重。